In [1]:
import openai
import re
import time
import json

import numpy as np

from tqdm import tqdm
from pprint import pprint
from tenacity import retry, stop_after_attempt, wait_chain, wait_fixed

import os
from openai import AzureOpenAI

import math

import re
import math
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import traceback

In [2]:
endpoint = "https://pankajaiml.openai.azure.com/"
model_name = "gpt-4o"
deployment = "gpt-4o"
subscription_key = "REDACTED_AZURE_OPENAI_KEY"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# Retry logic
@retry(wait=wait_chain(*[wait_fixed(3) for _ in range(3)] +
                       [wait_fixed(5) for _ in range(2)] +
                       [wait_fixed(10)]))
def completion_with_backoff(messages):
    return client.chat.completions.create(
        messages=messages,
        max_tokens=1512,
        temperature=0.0,
        model=deployment
    )

In [3]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as reader:
        data = json.load(reader)  # Load the entire JSON file
    return data

dev_data = load_json('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/testingDatasets/ASDIVsampled_train.json')
CoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/CoT_prompt_examples.txt').read()
Standard_prompt_examples = open("/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/standard_prompt_examples.txt").read()
CCoT_prompt_examples = open("/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/CCoT_prompt_example.txt").read()

In [5]:
# === Metrics ===
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/ASDIV/CoT.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        num = float(cleaned)
        return num
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(d):
    try:
        q = d['body'] +' '+ d['question']
        a = float(re.search(r'(\d+\.?\d*)', d['answer']).group(1))  # Ground truth

        prompt_q = (
            CoT_prompt_examples +
            '\nQ: ' + q + " Think step by step. Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions accurately."},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract Answer
        match = re.search(
            r'(?:the answer is|final answer:)\s*\**\$?(-?\d+(?:\.\d+)?)\**\s*(?:[a-zA-Z%$ ]+)?[\.]?',
            ans_model,
            re.IGNORECASE
        )
        if match:
            extracted_raw = match.group(1).strip()
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Log Block
        log_block = (
            f'Q: {q}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted:\n{extracted}\n'
            f'A:\n{a}\n\n'
        )

        # === Determine Correctness
        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ Incorrect or Invalid\n" + log_block

    except Exception as e:
        return "error", f"Error processing entry: {d}\nException: {str(e)}\n\n"

# === Main Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type == "incorrect":
                bad_fd.write(log)
            elif result_type == "error":
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  0%|          | 1/205 [00:01<06:42,  1.98s/it]

Accuracy: 1 / 1 = 100.00%
Accuracy: 2 / 2 = 100.00%


  1%|▏         | 3/205 [00:04<05:22,  1.60s/it]

Accuracy: 3 / 3 = 100.00%
Accuracy: 4 / 4 = 100.00%
Accuracy: 5 / 5 = 100.00%
Accuracy: 6 / 6 = 100.00%
Accuracy: 7 / 7 = 100.00%
Accuracy: 7 / 8 = 87.50%
Accuracy: 8 / 9 = 88.89%
Accuracy: 9 / 10 = 90.00%
Accuracy: 10 / 11 = 90.91%
Accuracy: 11 / 12 = 91.67%
Accuracy: 12 / 13 = 92.31%


  7%|▋         | 14/205 [00:06<01:05,  2.90it/s]

...

Accuracy: 177 / 188 = 94.15%


 92%|█████████▏| 189/205 [08:08<00:53,  3.37s/it]

Accuracy: 178 / 189 = 94.18%
Accuracy: 179 / 190 = 94.21%
Accuracy: 180 / 191 = 94.24%
Accuracy: 181 / 192 = 94.27%
Accuracy: 182 / 193 = 94.30%
Accuracy: 183 / 194 = 94.33%
Accuracy: 184 / 195 = 94.36%
Accuracy: 185 / 196 = 94.39%
Accuracy: 186 / 197 = 94.42%
Accuracy: 186 / 198 = 93.94%
Accuracy: 187 / 199 = 93.97%
Accuracy: 188 / 200 = 94.00%
Accuracy: 189 / 201 = 94.03%
Accuracy: 190 / 202 = 94.06%
Accuracy: 191 / 203 = 94.09%


100%|██████████| 205/205 [08:09<00:00,  2.39s/it]

Accuracy: 191 / 204 = 93.63%
Accuracy: 192 / 205 = 93.66%


In [5]:
# === Metrics ===
acc = 0
total = 0

# === File Output Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/ASDIV/standard.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)  # Keep digits, decimal, minus
    try:
        num = float(cleaned)
        return round(num, 4)
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(d):
    try:
        q = d['body'] +' '+ d['question']
        a = float(re.search(r'(\d+\.?\d*)', d['answer']).group(1))  # Ground truth

        prompt_q = (
            Standard_prompt_examples +
            '\nAnswer this question: ' + q + " Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions correctly."},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Improved Answer Extraction ===
        match = re.search(r'the answer is\s*([-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Log Block
        log_block = (
            f'Q: {q}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted:\n{extracted}\n'
            f'A:\n{a}\n\n'
        )

        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ Incorrect or Invalid\n" + log_block

    except Exception as e:
        return "error", f"Error processing entry: {d}\nException: {str(e)}\n\n"

# === Main Parallel Processing ===
results = []
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                global acc
                acc += 1
                fd.write(log)
            elif result_type == "incorrect":
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")
    fd.write(f"\nFinal Accuracy: {acc} / {total} = {acc / total:.2%}\n")
    fd.write(f"Final Accuracy: {acc} / {total} = {acc / total:.2%}\n") 

  1%|▏         | 3/205 [00:02<02:01,  1.67it/s]

Accuracy: 1 / 1 = 100.00%
Accuracy: 2 / 2 = 100.00%
Accuracy: 3 / 3 = 100.00%
Accuracy: 4 / 4 = 100.00%
Accuracy: 5 / 5 = 100.00%
Accuracy: 6 / 6 = 100.00%
Accuracy: 7 / 7 = 100.00%
Accuracy: 8 / 8 = 100.00%


  4%|▍         | 9/205 [00:02<00:41,  4.71it/s]

Accuracy: 9 / 9 = 100.00%
Accuracy: 10 / 10 = 100.00%
Accuracy: 11 / 11 = 100.00%
Accuracy: 12 / 12 = 100.00%
Accuracy: 13 / 13 = 100.00%


  7%|▋         | 14/205 [00:07<01:54,  1.67it/s]

...

Accuracy: 171 / 196 = 87.24%


 96%|█████████▌| 197/205 [05:04<00:21,  2.75s/it]

Accuracy: 172 / 197 = 87.31%
Accuracy: 172 / 198 = 86.87%
Accuracy: 173 / 199 = 86.93%
Accuracy: 174 / 200 = 87.00%
Accuracy: 175 / 201 = 87.06%
Accuracy: 176 / 202 = 87.13%
Accuracy: 177 / 203 = 87.19%


100%|██████████| 205/205 [05:06<00:00,  1.50s/it]

Accuracy: 177 / 204 = 86.76%
Accuracy: 178 / 205 = 86.83%


In [4]:
# === Metrics ===
acc = 0
total = 0

# === File Output Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/ASDIV/complexCoT.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')
error_log_path = output_path.replace('.txt', '_errors.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        num = float(cleaned)
        return round(num, 4)
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(idx, d):
    global acc, total
    try:
        q = d['body'] +' '+ d['question']
        a = float(re.search(r'(\d+\.?\d*)', d['answer']).group(1))  # Ground truth
        
        # === Prompt Setup for Complex CoT ===
        prompt_q = (
            CCoT_prompt_examples +
            "\nQ: " + q + "\n\n"
            "Please reason through this problem using a complex, multi-step chain of thought:\n"
            "Step 1: Clearly state all given information and any assumptions.\n"
            "Step 2: Propose two different methods to solve the problem, briefly outlining the logic of each.\n"
            "Step 3: For each method, work through all intermediate steps in detail, showing calculations, checks, and potential pitfalls.\n"
            "Step 4: Evaluate and compare the two methods—discussing which is better based on clarity, reliability, or efficiency.\n"
            "Step 5: Choose the better method and use it to solve the problem, showing all steps.\n"
            "Step 6: Double-check the solution for errors or unreasonable results.\n"
            "Finish your response with: the answer is <answer>"
        )

        messages = [
            {
                "role": "system",
                "content": (
                    "Your goal is to answer the question using a complex, coherent, step by step thoughts, answering the questions correctly. Write your final answer as: The answer is <answer>.\n"
                )
            },
            {"role": "user", "content": prompt_q}
        ]

        # === Get Response ===
        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Improved Answer Extraction ===
# === Improved Answer Extraction ===
        match = re.search(
            r'(?:the answer is|final answer:)\s*\**\$?(-?\d+(?:\.\d+)?)\**\s*(?:[a-zA-Z%$ ]+)?[\.]?',
            ans_model,
            re.IGNORECASE
        )
        if match:
            extracted_raw = match.group(1).strip()
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None


        # === Log Block
        log_block = (
            f'Q: {q}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted:\n{extracted}\n'
            f'A:\n{a}\n\n'
        )

        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ Incorrect or Invalid\n" + log_block

    except Exception as e:
        # Log the error and the problematic entry
        error_log = f"Error at index {idx}:\nData: {d}\nTraceback:\n{traceback.format_exc()}\n\n"
        return "error", error_log

# === Main Parallel Processing ===
results = []
start_index = 154  # Start processing from question 127
with open(output_path, 'a') as fd, open(bad_output_path, 'a') as bad_fd, open(error_log_path, 'a') as error_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, idx, d) for idx, d in enumerate(dev_data[start_index:], start=start_index)]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type == "incorrect":
                bad_fd.write(log)
            elif result_type == "error":
                error_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

    # Final accuracy report
    fd.write(f"\nFinal Accuracy: {acc} / {total} = {acc / total:.2%}\n")
    fd.write(f"Final Accuracy: {acc} / {total} = {acc / total:.2%}\n")  # Write to the output file


  2%|▏         | 1/51 [00:09<08:11,  9.83s/it]

Accuracy: 1 / 1 = 100.00%


  4%|▍         | 2/51 [00:11<04:11,  5.13s/it]

Accuracy: 2 / 2 = 100.00%
Accuracy: 3 / 3 = 100.00%
Accuracy: 4 / 4 = 100.00%
Accuracy: 5 / 5 = 100.00%
Accuracy: 6 / 6 = 100.00%
Accuracy: 7 / 7 = 100.00%
Accuracy: 8 / 8 = 100.00%
Accuracy: 9 / 9 = 100.00%


 20%|█▉        | 10/51 [00:12<00:30,  1.33it/s]

...

Accuracy: 37 / 39 = 94.87%
Accuracy: 38 / 40 = 95.00%
Accuracy: 39 / 41 = 95.12%
Accuracy: 40 / 42 = 95.24%


 84%|████████▍ | 43/51 [02:13<00:25,  3.23s/it]

Accuracy: 41 / 43 = 95.35%
Accuracy: 41 / 44 = 93.18%
Accuracy: 42 / 45 = 93.33%
Accuracy: 43 / 46 = 93.48%
Accuracy: 44 / 47 = 93.62%
Accuracy: 45 / 48 = 93.75%
Accuracy: 46 / 49 = 93.88%


100%|██████████| 51/51 [02:17<00:00,  2.70s/it]

Accuracy: 46 / 50 = 92.00%
Accuracy: 47 / 51 = 92.16%
